<table>
  <tr>
    <td><div align="left"><font size="30">Discrete-time and hybrid systems</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In this notebook we will build a model of a simple discrete-time system, a first-order system with a feedback loop, and simulate it.

Basic knowledge of `bdsim` from the "Getting started" notebook is assumed.

We start with the standard boilerplate: import the package, establish the simulation environment, create an empty block diagram to work with.

In [ ]:
import bdsim

sim = bdsim.BDSim()
bd = sim.blockdiagram()

# Discrete-time model

We will create a discrete-time equivalent model of the system used in the "Getting started" notebook which was

![Block diagram sketch](https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bd1-sketch.png)

For a discrete-time system we must first create a clock source

In [ ]:
clock = bd.clock(10, "Hz")

`bdsim` does not have functionality to convert a continuous-time transfer function to discrete-time, but there are lots of packages that will do it for you (the first three are open source)


* Python Control [`control.sample_system`](https://python-control.readthedocs.io/en/latest/generated/control.sample_system.html)
* SciPy [`scipy.signal.cont2discrete`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.cont2discrete.html)
* Octave [`tf`](https://gnu-octave.github.io/pkg-control/@tf_tf.html) and [`c2d`](https://gnu-octave.github.io/pkg-control/@lti_c2d.html)
* MATLAB `tf` and `c2d`

For a sample rate of 10Hz and using the ZOH method for discretisation, we can write a discrete-time version of the plant as follows:

In [ ]:
plant = bd.LTI_SISO_S(clock, 0.02439, [1, -0.9512], name="plant")

Note that in the continuous time case the block name was `LTI_SISO`.  


<div class="alert alert-block alert-info">
<b>Note:</b> All <code>bdsim</code> discrete-time blocks have a <code>_S</code> suffix which denotes "<strong>S</strong>ampled data".  All such blocks must have a clock as their first argument.
</div>




Now let's define the remaining blocks

In [ ]:
sum = bd.SUM("+-")
gain = bd.GAIN(10)
demand = bd.STEP(T=1, name="demand")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right", stairs=True)

One very subtle difference is the argument `stairs=True` to the `SCOPE` block.  Since this is a discrete-time system we should properly consider the signals as piecewise constant with respect to time.

In [ ]:
bd.connect(sum, gain)
bd.connect(plant, sum[1])
bd.connect(demand, sum[0], scope[1])
bd.connect(gain, plant)
bd.connect(plant, scope[0])

and we could also use the Pythonic operator overload approach introduced in the "Getting started" notebook.


Next we compile and check our model

In [ ]:
bd.compile(verbose=True)
bd.report_summary()


and then run it.  We'll see what happens over the first 5 seconds.

In [ ]:
out = sim.run(bd, T=5)

We see a classic first-order response to the unit step at time $t=1$ and considerable steady state error.

The time series data from the simulation is available in the return value `out` but the format is a little different this time

In [ ]:
print(out)

`t` and `x` represent the continuous-time state but is empty in this case, since there are no continuous time states.

Instead we look at the substructure for the clock object, `clock0` which contains a time vector `t` with 50 elements (5s at 10Hz) and a state vector with single column. According to `clock0.xnames` that column  corresponds to "plant:X_0", the only state variable of our first-order plant.

<div class="alert alert-block alert-info">
<b>Note:</b> Note that our clock was assigned a default name of <code>clock.0</code> and the struct object holding our results uses a dot to separate fields.  Therefore the name of the clock field in this case is <code>clock0</code> not <code>clock.0</code>.
</div>

<div class="alert alert-block alert-info">
<b>Note:</b> Note that a system can contain multiple clocks with different frequencies and relative offsets.  Each sampled-data block is associated with one of those clocks, and its state is stored with that clock.
</div>

# Hybrid model


We will simulate a hybrid model that comprises continuous- and discrete-time dynamic components. We will start with the continuous-time system from the "Getting started" notebook, but insert a zero-order hold (ZOH) to discretise the controller output (between the `GAIN` and the `LTI_SS` blocks).

![Block diagram sketch](https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bd1-sketch%2Bzoh.png)

The ZOH will be clocked at 10Hz.  Here's the `bdsim` model


In [ ]:
bd = sim.blockdiagram()

clock = bd.clock(5, "Hz")

zoh = bd.ZOH(clock)

plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
sum = bd.SUM("+-")
gain = bd.GAIN(10)
demand = bd.STEP(T=1, name="demand")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

bd.connect(sum, gain)
bd.connect(plant, sum[1])
bd.connect(demand, sum[0], scope[1])
bd.connect(plant, scope[0])

# replace gain -> plant with gain -> zoh -> plant
# bd.connect(gain, zoh)
bd.connect(gain, zoh)
bd.connect(zoh, plant)

bd.compile()

out = sim.run(bd, T=5)

The result looks similar to the continuous-time system in the "Getting started" notebook.  If we look at the simulation results we can see both the discrete-time and continuous time states

In [ ]:
print(out)

There is one continuous-time state vector `x[:,0]` corresponding to the plant, and one discrete-time state vector `clock1.X[:,0]` corresponding to the ZOH.  Each state vector has a corresponding time vector `t` and `clock1.t`.  

<div class="alert alert-block alert-info">
<b>Note:</b>The elements of <code>clock1.t</code> are regularly spaced because that subsystem is driven by a 5Hz clock.  The elements of <code>t</code> will be non-uniformly spaced as determined by the numerical integrator.  Regular sampling can be achieved by passing the <code>dt</code> option to <code>sim.run(...)</code>.
</div>


